In [ ]:
import gym 
import numpy as np
import time
import matplotlib.pyplot as plt
import random
from tqdm.notebook import tqdm

# Q-Learning (Simple Overview)

## 1. Goal
Learn the **optimal action-value function** $Q^*(s,a)$ that tells us the *expected discounted return* if we take action $a$ in state $s$ and then act optimally.

## 2. Core Idea
Instead of learning a policy directly, Q-Learning learns numeric values for each state–action pair. We update those values toward a **bootstrapped target** that uses the *best* next action according to current estimates.

## 3. Update Rule
For a transition $(s,a,r,s',\text{done})$:
$
Q(s,a)\leftarrow Q(s,a) + \alpha\Bigl[r + \gamma (1-\text{done}) \max_{a'} Q(s',a') - Q(s,a)\Bigr]
$

**Terms:**
- $\alpha$: learning rate (step size)
- $\gamma$: discount factor (how much we value future rewards)
- done: 1 if episode ended; then no future term
- TD Error $= \text{target} - Q(s,a)$

## 4. Exploration vs Exploitation (ε-Greedy)
Choose:
- Random action with probability $\varepsilon$ (exploration)
- Action $\arg\max_a Q(s,a)$ with probability $1 - \varepsilon$ (exploitation)

Gradually **decay** $\varepsilon$ from a high value (e.g. 1.0) to a small floor (e.g. 0.01).

In [ ]:
#Load the gym env
env = gym.make("CartPole-v1")

In [ ]:
#Define state bins 
upperBounds=env.observation_space.high
lowerBounds=env.observation_space.low
state_bins = [np.linspace(lowerBounds[0],upperBounds[0], 10),
              np.linspace(lowerBounds[1],upperBounds[1], 10),
              np.linspace(lowerBounds[2],upperBounds[2], 10),
              np.linspace(lowerBounds[3],upperBounds[3], 10),]

#Discretize 
def discretize_state(observation,state_bins):
    state=[]
    try:
        obs_array = observation[0]
        for i in range(obs_array.shape[0]):
            state.append(np.digitize(np.array(obs_array[i]), state_bins[i])-1)
    except:
        for i in range (len(observation)):
            state.append(np.digitize(np.array(observation[i]),state_bins[i])-1)
    return tuple(state)

In [ ]:
alpha = 0.9
gamma = 0.99
epsilon= 0.9
epsilon_decay = 0.99
episodes= 2000
timesteps= 500
arr=[]
Q = np.zeros((len(state_bins[0]),
            len(state_bins[1]),
            len(state_bins[2]),
            len(state_bins[3]),2)
            ,dtype=np.float32)
for e in tqdm(range(episodes)):
    initial_state = env.reset()
    observation = initial_state[0]
    if epsilon > 0.01:
        epsilon = epsilon*epsilon_decay
    if e % 100 == 0:
        if alpha >  0.001:
            alpha = alpha * 0.97
    sum_reward= 0
    for i in range(timesteps):
        p= np.random.rand()
        discrete_state = discretize_state(observation,state_bins)
        if p <= epsilon:
            action = random.choice([0, 1])
        elif p > epsilon:
            action = np.argmax(Q[discrete_state])
        next_observation, reward, terminated, truncated, info = env.step(action)
        next_discrete_state = discretize_state(next_observation, state_bins)
        # Update Q-table using Q-learning update rule
        if (terminated):
            Q[discrete_state][action] = (1 - alpha) * Q[discrete_state][action] + alpha * (reward)
            arr.append(sum_reward)
            break
        else :
            next_best_action= np.argmax(Q[next_discrete_state])
            Q[discrete_state][action] = (1 - alpha) * Q[discrete_state][action] + alpha * (reward + gamma * Q[next_discrete_state][next_best_action])
            sum_reward += reward
        observation = next_observation
env.close()
plt.scatter(range(len(arr)), arr)
plt.show()
arr = []

In [ ]:
episodes = 1
env=gym.make('CartPole-v1', render_mode="human")
total_reward = 0 
timesteps= 1000
img_array = []
for ei in range (episodes):
    initial_state = env.reset()
    observation = initial_state[0]
    total_reward = 0
    ti = 0 
    while True:
        ti+= 1
        discrete_state = discretize_state(observation,state_bins)
        action = np.argmax(Q[discrete_state])
        next_observation, reward, terminated, truncated, info = env.step(action)
        next_discrete_state = discretize_state(next_observation, state_bins)
        observation = next_observation
        total_reward += reward
        if (terminated):
            print("Episode: ", ei, "Time: ", ti, "Reward: ",total_reward)
            break
env.close()
